# Création de la base de données

## Les données utilisées

Dans le cadre de ce projet, l'objectif est d'évaluer l'efficacité réelle du Pass Culture. Pour ce faire, nous utilisons différentes bases de données.

In [85]:
import pandas as pd
from functions import *

## Basilic

la Base des lieux et équipements culturels (Basilic) est une base de données réalisée par agrégation de différentes sources : bases de la direction générale des patrimoines et de l’architecture, de la direction générale de la création artistique, de la direction générale des médias et des industries culturelles, de la délégation générale à la transmission, aux territoires et à la démocratie culturelle, du Centre national du cinéma et de l'image animée, du Centre national du livre, du Centre national des arts du cirque, de la rue et du théâtre (Artcena), de la Médiathèque du patrimoine et de la photographie.

Elle porte sur le champ de la France entière, et va nous permettre de trouver toutes les infrastructures culturelles.

In [86]:
# Importation des données

df_brut_basilic = importdata("basilic_29072026.csv", ";")
infosbase(df_brut_basilic)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 86366
Nombre de colonnes  : 54

NOMS DES COLONNES ET TYPES
Nom                                         object
Adresse                                     object
Complement Adresse                         float64
Code Postal                                 object
libelle_geographique                        object
code_insee                                  object
Code Insee Arrondt                          object
Identifiant origine                         object
Type équipement ou lieu                     object
Label et appellation                        object
Région                                      object
Domaine                                     object
Archéologie détail                          object
Adresse postale                             object
Département                                 object
Précision équipement                        object
N_Département                               object
N_Région               

/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (3,12,20,21,22,28,31,43) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


Voici maintenant les informations générales sur la base, préalables au nettoyage.

In [87]:
print("\n" + "=" * 60)
print("VALEURS MANQUANTES PAR COLONNE")
print("=" * 60)
print(df_brut_basilic.isna().sum())


VALEURS MANQUANTES PAR COLONNE
Nom                                            0
Adresse                                    31397
Complement Adresse                         86366
Code Postal                                    9
libelle_geographique                           0
code_insee                                     0
Code Insee Arrondt                            85
Identifiant origine                         3013
Type équipement ou lieu                        0
Label et appellation                       30483
Région                                        12
Domaine                                        0
Archéologie détail                         85804
Adresse postale                                0
Département                                   12
Précision équipement                       65851
N_Département                                  9
N_Région                                      12
Fonction_1                                  7372
Fonction_2                           

#### Choix des variables

Maintenant, nous allons nettoyer la base, pour ne garder que les variables qui sont pertinentes pour notre sujet.

Ainsi, nous allons garder quelques variables permettant d'identifier l'infrastructure de manière unique ("Nom", "libelle_geographique", "code_insee", "Code Insee Arrondt", "Label et appellation", "Région", "Département", "Fonction_1", "Fonction_2", "Fonction_3", "Fonction_4", "Type_de_cinema", "Nombre_ecrans", "Nombre_de_salles_de_theatre", "Surface_Bibliotheque", "Precision_protection_sites_et_monuments").

In [88]:
df_basilic = df_brut_basilic[["Nom", 
    "libelle_geographique",
    "Département",
    "Région",  
    "code_insee", 
    "Code Insee Arrondt", 
    "Label et appellation", 
    "Fonction_1", 
    "Fonction_2", 
    "Fonction_3", 
    "Fonction_4", 
    "Type_de_cinema", 
    "Nombre_ecrans", 
    "Nombre_de_salles_de_theatre", 
    "Surface_Bibliotheque", 
    "Precision_protection_sites_et_monuments"]]

Nous allons maintenant transformer la base pour que la granularité ne soit pas au niveau du batiment, mais au niveau de la ville : nous aurons ainsi une ligne par commune, et pour chaque type d'équipement, le nombre d'établissements, et le cas échéant, le nombre total de salles ou la surface totale (cinéma, théâtre, bibliothèque...).

La première étape est de vérifier la colonne à partir de laquelle nous allons compter le nombre de bâtiments.

In [89]:
print("Label et appellation", df_basilic["Label et appellation"].unique())

Label et appellation ['Monument historique' nan 'Centre d’art contemporain d’intérêt national'
 'Centre chorégraphique national'
 'Centre de développement chorégraphique national'
 'Centre dramatique national' 'Art et essai'
 'Centre culturel de rencontre' 'Scène conventionnée d’intérêt national'
 'Scène nationale' 'Scène de musiques actuelles'
 'Site patrimonial remarquable' 'Théâtre hors label' 'Théâtre national'
 'Théâtre privé' 'Théâtre de ville' "Patrimoine mondial de l'Unesco"
 'Microfolie' 'Centre national des arts de la rue et de l’espace public'
 'Centre national de création musicale' 'Compagnie avec lieu'
 'Compagnie subventionnée (DRAC - Aide à la production)'
 'Compagnie conventionnée' "Fonds régional d'art contemporain"
 'Itinéraire culturel européen' 'Jardin remarquable'
 'LIR - Librairie indépendante de référence' 'LR - Librairie de référence'
 'Maison des illustres' 'Monument national' 'Musée de France'
 'Orchestre national en région' 'Pôle national du cirque'
 'Archite

Maintenant, nous pouvons compter le nombre de bâtiment par ville, pour chaque type et au total.

In [90]:
# Indicatrices, avec les NaN regroupés dans "autre_etablissements"
# ici on commence par vérifier qu'on n'a pas déjà une colonne autre_etablissement
assert "autre_etablissements" not in df_basilic["Label et appellation"].unique()
label_dummies = pd.get_dummies(
    df_basilic["Label et appellation"].fillna("autre_etablissements")
)

# Agrégation par commune
agg_labels = label_dummies.groupby(df_basilic["code_insee"]).sum()

# Agrégats numériques
agg_numeriques = df_basilic.groupby("code_insee").agg(
    nb_ecrans_total=("Nombre_ecrans", "sum"),
    nb_salles_theatre_total=("Nombre_de_salles_de_theatre", "sum"),
    surface_bibliotheque_total=("Surface_Bibliotheque", "sum"),
    nb_etablissements=("Nom", "count")
)

# Infos communales
group_keys = ["code_insee", "libelle_geographique", "Département", "Région"]
infos_commune = df_basilic[group_keys].drop_duplicates(subset="code_insee").set_index("code_insee")

# Fusion finale
df_communes = infos_commune.join([agg_labels, agg_numeriques]).reset_index()

In [91]:
# Informations sur la base de données

infosbase(df_communes)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 22084
Nombre de colonnes  : 49

NOMS DES COLONNES ET TYPES
code_insee                                                   object
libelle_geographique                                         object
Département                                                  object
Région                                                       object
Architecture contemporaine remarquable                        int64
Art et essai                                                  int64
Centre chorégraphique national                                int64
Centre culturel de rencontre                                  int64
Centre de développement chorégraphique national               int64
Centre dramatique national                                    int64
Centre d’art contemporain d’intérêt national                  int64
Centre national de création musicale                          int64
Centre national de la marionnette                             int64
Centre

A ce stade, la base de données recense de manière très exhaustive les équipements de la commune en terme de culture. Le but est maintenant de regrouper les établissements en groupe d'établissements similaires afin de produire des données plus lisibles.

<table>
<tr><th>Nouvelle variable</th><th>Anciennes variables</th></tr>
<tr><td rowspan="8">Patrimoine</td><td>Architecture contemporaine remarquable"
<tr><td>Monument historique</td></tr>
<tr><td>Monument national</td></tr>
<tr><td>Site patrimonial remarquable</td></tr>
<tr><td>Patrimoine mondial de l'Unesco</td></tr>
<tr><td>Jardin remarquable</td></tr>
<tr><td>Maison des illustres</td></tr>
<tr><td>Itinéraire culturel européen</td></tr>
<tr><td rowspan="8">Centre culturel national</td><td>Centre chorégraphique national</td></tr>
<tr><td>Centre culturel de rencontre</td></tr>
<tr><td>Centre de développement chorégraphique national</td></tr>
<tr><td>Centre dramatique national</td></tr>
<tr><td>Centre national de création musicale</td></tr>
<tr><td>Centre national de la marionnette</td></tr>
<tr><td>Centre national des arts de la rue et de l'espace public</td></tr>
<tr><td>Pôle national du cirque</td></tr>
<tr><td rowspan="5">Musée et Art contemporain</td><td>Musée national</td></tr>
<tr><td>Musée de France</td></tr>
<tr><td>Centre d'art contemporain d'intérêt national</td></tr>
<tr><td>Fonds régional d'art contemporain</td></tr>
<tr><td>Microfolie</td></tr>
<tr><td rowspan="7">Théâtre</td><td>Théâtre national</td></tr>
<tr><td>Théâtre hors label</td></tr>
<tr><td>Théâtre privé</td></tr>
<tr><td>Théâtre de ville</td></tr>
<tr><td>Scène conventionnée d'intérêt national</td></tr>
<tr><td>Scène nationale</td></tr>
<tr><td>Théâtre lyrique d'intérêt national</td></tr>
<tr><td rowspan="5">Concert et Musique</td><td>Zénith</td></tr>
<tr><td>Scène de musiques actuelles</td></tr>
<tr><td>Orchestre national en région</td></tr>
<tr><td>Opéra national</td></tr>
<tr><td>Opéra national en région</td></tr>
<tr><td rowspan="1">Autre</td><td>NaN</td></tr>


In [92]:
nouvelles_var = ["Autre",
    "Centre culturel national",
    "Compagnie",
    "Concert et Musique",
    "Etablissement public national",
    "Librairie",
    "Musée et Art contemporain",
    "Patrimoine", 
    "Théâtre"]

groupes = {
    "Librairie" : [
        "LIR - Librairie indépendante de référence", 
        "LR - Librairie de référence"],
    "Compagnie" : [
        "Compagnie avec lieu",
        "Compagnie conventionnée",
        "Compagnie subventionnée (DRAC - Aide à la production)"],
    "Patrimoine": [
        "Architecture contemporaine remarquable", 
        "Monument historique",
        "Monument national",
        "Site patrimonial remarquable",
        "Patrimoine mondial de l'Unesco",
        "Jardin remarquable","Maison des illustres",
        "Itinéraire culturel européen"],
    "Centre culturel national" : [
        "Centre chorégraphique national",
        "Centre culturel de rencontre",
        "Centre de développement chorégraphique national",
        "Centre dramatique national",
        "Centre national de création musicale",
        "Centre national de la marionnette",
        "Centre national des arts de la rue et de l’espace public",
        "Pôle national du cirque"],
    "Musée et Art contemporain": [
        "Musée national",
        "Musée de France",
        "Centre d’art contemporain d’intérêt national",
        "Fonds régional d'art contemporain",
        "Microfolie"],
    "Théâtre":[
        "Théâtre national",
        "Théâtre hors label",
        "Théâtre privé",
        "Théâtre de ville",
        "Scène conventionnée d’intérêt national",
        "Scène nationale",
        "Théâtre lyrique d'intérêt national"],
    "Concert et Musique": [
        "Zénith",
        "Scène de musiques actuelles",
        "Orchestre national en région",
        "Opéra national",
        "Opéra national en région"],
    "Etablissement public national" : ["Etablissement public national"],
    "Autre":["autre_etablissements"]
}

colonnes_a_supprimer = [] #servira à nettoyer le df à la fin

for nv_var in nouvelles_var:
    df_communes[nv_var] = 0  # on initialise la colonne à 0 avant de sommer
    for anc_var in groupes[nv_var]:
        if anc_var not in df_communes.columns:
            print(f"Attention, Colonne absente : {anc_var}")
            continue
        df_communes[nv_var] += df_communes[anc_var]
        colonnes_a_supprimer.append(anc_var)

# on supprime les anciennes colonnes seulement à la fin, en une fois, avec axis=1
df_communes = df_communes.drop(columns=colonnes_a_supprimer)

In [93]:
infosbase(df_communes)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 22084
Nombre de colonnes  : 17

NOMS DES COLONNES ET TYPES
code_insee                     object
libelle_geographique           object
Département                    object
Région                         object
Art et essai                    int64
nb_ecrans_total               float64
nb_salles_theatre_total       float64
surface_bibliotheque_total    float64
nb_etablissements               int64
Autre                           int64
Centre culturel national        int64
Compagnie                       int64
Concert et Musique              int64
Librairie                       int64
Musée et Art contemporain       int64
Patrimoine                      int64
Théâtre                         int64
dtype: object


A ce stade, la base de données recense donc l'ensemble des équipements disponibles pour chaque commune. Nous allons maintenant y ajouter des informations complémentaires telles que des informations démographiques.

## Démographie et âges (INSEE)

Nous importons maintenant la base de données de l'INSEE permettant d'obtenir la population des communes par tranche d'âge de cinq ans et par sexe. Le but sera de joindre cette base avec les données issues de la base Basilic afin d'arriver à obtenir la densité d'équipements culturels pas habitant.

Dans un premier temps, nous importons les données. Comme pour la base Basilic, elles ont été téléchargées (le 30 juillet 2026) et stockées localement.

In [ ]:
# Importation des données
df_brut_pop = importdata("DS_RP_TD_POPULATION_AGEHARSEX_PRINC_2023_data.csv", ";")

In [ ]:
# Nettoyage de la base : seulement les variables qui nous intéressent et la population qui nous intéresse
df_pop = df_brut_pop[
    (df_brut_pop['AGE'].isin(['Y15T19', 'Y20T24'])) & (df_brut_pop['GEO_OBJECT'] == 'COM')
]
df_pop = df_pop[["GEO",
    "AGE",
    "HAR",
    "SEX",
    "FREQ",
    "OBS_STATUS",
    "OBS_VALUE"]]
infosbase(df_pop)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 2091480
Nombre de colonnes  : 7

NOMS DES COLONNES ET TYPES
GEO            object
AGE            object
HAR            object
SEX            object
FREQ           object
OBS_STATUS     object
OBS_VALUE     float64
dtype: object


## Revenus et niveau de vie (INSEE - Filosofi)

Nous allons maintenant importer les données de revenu et de niveau de vie qui pourront servir de variables de contrôle ou d'instrument dans l'analyse de l'influence du Pass Culture sur les pratiques culturelles des jeunes.

In [ ]:
df_brut_filosofi = importdata("DS_FILOSOFI_CC_2023_data.csv", ";")
df_filosofi = df_brut_filosofi[(df_brut_filosofi["GEO_OBJECT"] == "COM")]
infosbase(df_brut_filosofi)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 1123173
Nombre de colonnes  : 9

NOMS DES COLONNES ET TYPES
FILOSOFI_MEASURE     object
GEO                  object
GEO_OBJECT           object
UNIT_MEASURE         object
CONF_STATUS          object
OBS_STATUS           object
UNIT_MULT             int64
TIME_PERIOD           int64
OBS_VALUE           float64
dtype: object


/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


In [ ]:
print("FILOSOFI_MEASURE", df_brut_filosofi["FILOSOFI_MEASURE"].unique())

FILOSOFI_MEASURE ['S_EI_DI_UNE' 'D3_SL' 'D2_SL' 'IQR_SL' 'S_RET_PEN_DI' 'D6_SL'
 'S_EI_DI_N_SAL' 'GI_SL' 'IR_D9_D1_SL' 'Q3_SL' 'D8_SL' 'Q1_SL' 'D7_SL'
 'S_SOC_BEN_DI_MIN_SOC' 'D1_SL' 'S_EI_DI_SAL' 'D9_SL' 'S_INC_ASS_DI'
 'S_EI_DI' 'S_SOC_BEN_DI' 'S80S20_SL' 'S_SOC_BEN_DI_HOU_BEN'
 'S_DIR_TAX_DI' 'S_SOC_BEN_DI_FAM_BEN' 'D4_SL' 'PR_MD60' 'MED_SL']


## Niveau de diplôme, éducation (INSEE)

Enfin, il s'agit d'importer les données sur le niveau de diplôme des individus, au niveau communal afin de pouvoir le comparer aux autres variables obtenues dans les autres bases de données.

In [ ]:
# On importe les données pour le niveau communal
df_brut_diplome = importdata("DS_RP_DIPLOMES_PRINC_2023_data.csv", ";")
df_diplome = df_brut_diplome[(df_brut_diplome["GEO_OBJECT"] == "COM")]
infosbase(df_brut_diplome)

/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 3257982
Nombre de colonnes  : 10

NOMS DES COLONNES ET TYPES
GEO             object
GEO_OBJECT      object
AGE             object
SEX             object
EDUC            object
RP_MEASURE      object
FREQ            object
OBS_STATUS      object
TIME_PERIOD      int64
OBS_VALUE      float64
dtype: object


Maintenant, nous allons sélectionner la population qui nous intéresse, et donner à la base un format qui nous permettra de la joindre aux autres bases préparées en amont.